# Integration Test - Document Intelligence Model

**Purpose**: End-to-end integration test with REAL Azure services and PDF documents

**Requirements**:
* Valid Azure OpenAI credentials in .env file
* Valid Azure Document Intelligence credentials in .env file
* Test PDF uploaded to: `/Volumes/datascience_dev_bronze_sandbox/ds_document_deidentification/test_documents/test_employee_medicare.pdf`

**Test Document**:
* Filename: `test_employee_medicare.pdf`
* Contents: Employee name and Medicare number
* Expected entities to detect:
  - Employee name (e.g., "Sarah Johnson")
  - Medicare number (e.g., "2428 56781 3")

**⚠️ WARNING**: This test calls REAL Azure APIs and will incur costs.

In [0]:
%pip install -r requirements.txt
dbutils.library.restartPython()

In [0]:
# import logging

# logging.basicConfig(level=logging.INFO)

In [0]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import time

# Add databricks module to path
databricks_path = "/Workspace/Users/Conny.GUNADI@suncorp.com.au/document_deidentification_tool/src/"
if databricks_path not in sys.path:
    sys.path.insert(0, databricks_path)

# Load environment variables from .env file
env_path = os.path.join(databricks_path, "privasee/databricks/.env")
load_dotenv(env_path)

print("✓ Libraries imported")
print(f"✓ Environment loaded from: {env_path}")

os.environ['AZURE_OPENAI_API_KEY'] = dbutils.secrets.get(scope='openai_00010_1', key='apikey')
os.environ['PROXY_CLIENT_ID'] = dbutils.secrets.get(scope="nginx_proxy_sp", key="client_id")
os.environ['PROXY_CLIENT_SECRET'] = dbutils.secrets.get(scope="nginx_proxy_sp", key="client_secret")
os.environ['DATABRICKS_TOKEN'] = dbutils.secrets.get(scope="Conny.GUNADI@suncorp.com.au", key="DATABRICKS_TOKEN_DEV")

# Verify required environment variables
required_vars = [
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "WORKSPACE_URL",
    "WORKSPACE_ID",
    "PROXY_PORT",
    "PROXY_ROUTE",
    "AZURE_DOCUMENT_INTELLIGENCE_KEY",
    "AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT",
    "UC_VOLUME_PATH",
    "DATABRICKS_HOST",
    "DATABRICKS_TOKEN",
]

missing_vars = [var for var in required_vars if not os.getenv(var)]
if missing_vars:
    print(f"\n❌ Missing required environment variables: {', '.join(missing_vars)}")
    raise EnvironmentError(f"Missing environment variables: {missing_vars}")
else:
    print(f"\n✓ All {len(required_vars)} required environment variables are set")

In [0]:
print("="*80)
print("INTEGRATION TEST CONFIGURATION")
print("="*80)

# Display configuration (mask sensitive keys)
print(f"\nAzure OpenAI Endpoint: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
print(f"Azure OpenAI API Key: {'*' * 20}")
print(f"Azure OpenAI Deployment: {os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')}")
print(f"Azure OpenAI API Version: {os.getenv('AZURE_OPENAI_API_VERSION')}")
print(f"Databricks workspace url: {os.getenv('WORKSPACE_URL')}")
print(f"Databricks workspace id: {os.getenv('WORKSPACE_ID')}")
print(f"Azure OpenAI Proxy Port: {os.getenv('PROXY_PORT')}")
print(f"Azure OpenAI Proxy Route: {os.getenv('PROXY_ROUTE')}")
print(f"\nAzure DI Endpoint: {os.getenv('AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT')}")
print(f"Azure DI API Key: {'*' * 20}")
print(f"\nVision Provider: {os.getenv('VISION_SERVICE_PROVIDER')}")
print(f"UC Volume Path: {os.getenv('UC_VOLUME_PATH')}")

# Test document location
test_doc_path = "/Volumes/datascience_dev_bronze_sandbox/ds_document_deidentification/test_documents"
print(f"\nTest Document Path: {test_doc_path}")
print("="*80)

In [0]:
from privasee.databricks.model.document_intelligence import DocumentIntelligenceModel

print("Initializing Document Intelligence Model...")
print("-" * 80)

# Initialize model
model = DocumentIntelligenceModel()

# Load context (initializes all services)
model.load_context(None)

# Verify services are initialized
print("\n✓ Model initialized successfully")
print(f"  - OCR Service: {type(model.ocr_service).__name__}")
print(f"  - Vision Service: {type(model.vision_service).__name__}")
print(f"  - Vision Provider: {model.vision_provider}")
print(f"  - BBox Matcher: {type(model.bbox_matcher).__name__}")
print(f"  - UC Volume Path: {model.uc_volume_path}")

In [0]:
# Define test parameters
TEST_PDF_FILENAME = "Sample GP document PDF.pdf"
TEST_DOCUMENT_PATH = "/Volumes/datascience_dev_bronze_sandbox/ds_document_deidentification/test_documents"
TEST_SESSION_ID = "integration_test_002"

# Construct full path
test_pdf_full_path = f"{TEST_DOCUMENT_PATH}/{TEST_PDF_FILENAME}"

print("Test Configuration:")
print("-" * 80)
print(f"PDF Filename: {TEST_PDF_FILENAME}")
print(f"Full Path: {test_pdf_full_path}")
print(f"Session ID: {TEST_SESSION_ID}")
print("-" * 80)

# Check if test file exists
if os.path.exists(test_pdf_full_path):
    file_size = os.path.getsize(test_pdf_full_path)
    print(f"\n✓ Test PDF found ({file_size:,} bytes)")
else:
    print(f"\n❌ WARNING: Test PDF not found!")
    print(f"  Please upload '{TEST_PDF_FILENAME}' to: {TEST_DOCUMENT_PATH}")

In [0]:
import requests

url = "https://suncorp-dev.cloud.databricks.com/api/2.0/fs/directories/Volumes/datascience_dev_bronze_sandbox/ds_document_deidentification/sessions/integration_test_002/"
headers = {'Authorization': f'Bearer {os.environ['DATABRICKS_TOKEN']}'}

response = requests.get(url, headers=headers)
response.json()

In [0]:
import pandas as pd
import json
import shutil
from pathlib import Path

print("="*80)
print("RUNNING DOCUMENT INTELLIGENCE PREDICTION")
print("="*80)

print(f"\nProcessing: {test_pdf_full_path}")
print("\nThis may take 10-30 seconds...")
print("-" * 80)

# Step 1: Create field definitions for target entities (as list, not JSON)
field_definitions = [
    {
        "name": "claimant_name",
        "description": "The name of the claimant of a personal injury claims. The claimant is the person who is injured and lodging a compensation claim.",
        "strategy": "Fake Data"
    },
    {
        "name": "incident_date",
        "description": "The date when an incident happened",
        "strategy": "Fake Data"
    }
]
print(f"✓ Field definitions created: {field_definitions}")

# Step 2: Upload PDF to UC volume session directory
# FIXED: Model expects files at {uc_volume_path}/{session_id}/original.{ext}
# NOT at {uc_volume_path}/sessions/{session_id}/original/{filename}
uc_volume_path = os.getenv('UC_VOLUME_PATH')
file_ext = Path(TEST_PDF_FILENAME).suffix  # Gets '.pdf'
session_dir = f"{uc_volume_path}/{TEST_SESSION_ID}"
os.makedirs(session_dir, exist_ok=True)

# Name file as 'original.pdf' (not in 'original/' subdirectory)
destination_path = f"{session_dir}/original{file_ext}"
try:
    shutil.copy2(test_pdf_full_path, destination_path)
    file_size = os.path.getsize(destination_path)
    print(f"✓ PDF uploaded to UC volume session directory ({file_size:,} bytes)")
    print(f"  Location: {destination_path}")
except FileNotFoundError:
    print(f"\n❌ ERROR: Source PDF file not found at {test_pdf_full_path}")
    print(f"\nPlease upload '{TEST_PDF_FILENAME}' to:")
    print(f"  {TEST_DOCUMENT_PATH}")
    raise
except Exception as e:
    print(f"\n❌ ERROR: Failed to copy file to UC volume: {str(e)}")
    raise

# Step 3: Create pandas DataFrame with new structure
# Model expects: session_id, field_definitions (list)
# Model will fetch the file using _fetch_original_file(session_id)
input_df = pd.DataFrame([{
    "session_id": TEST_SESSION_ID,
    "field_definitions": field_definitions
}])
print(f"✓ Input DataFrame created with {len(input_df)} row(s)")
print(f"  Columns: {list(input_df.columns)}")

print("\nStarting prediction...")
print("-" * 80)

# Measure execution time
start_time = time.time()

try:
    # Run prediction with properly formatted DataFrame
    result_df = model.predict(context=None, model_input=input_df)
    
    elapsed_time = time.time() - start_time
    
    print(f"\n✓ Prediction completed in {elapsed_time:.2f} seconds")
    
    # Extract result from DataFrame
    result = result_df.iloc[0].to_dict()
    
except Exception as e:
    print(f"\n❌ ERROR during prediction: {str(e)}")
    import traceback
    traceback.print_exc()
    raise

In [0]:
result

In [0]:
print(test_pdf_full_path, destination_path)

In [0]:
print("="*80)
print("PREDICTION RESULTS")
print("="*80)

# Display summary
entities = result.get("entities", [])
pages = result.get("pages", [])

print(f"\nTotal Entities Detected: {len(entities)}")
print(f"Total Pages Processed: {len(pages)}")
print(f"Processing Time: {elapsed_time:.2f} seconds")
print(f"Average Time per Page: {elapsed_time / max(len(pages), 1):.2f} seconds")

print("\n" + "-"*80)
print("DETECTED ENTITIES")
print("-"*80)

# Group entities by type
entity_groups = {}
for entity in entities:
    entity_type = entity.get("entity_type", "unknown")
    if entity_type not in entity_groups:
        entity_groups[entity_type] = []
    entity_groups[entity_type].append(entity)

# Display entities by type
for entity_type, entity_list in entity_groups.items():
    print(f"\n{entity_type.upper()} ({len(entity_list)} found):")
    for i, entity in enumerate(entity_list, 1):
        text = entity.get("text", "")
        confidence = entity.get("confidence", "N/A")
        page_num = entity.get("page_num", "N/A")
        bbox = entity.get("bounding_box", {})
        
        print(f"  {i}. '{text}'")
        print(f"     - Page: {page_num}")
        print(f"     - Confidence: {confidence}")
        if bbox:
            print(f"     - BBox: x={bbox.get('x', 'N/A'):.1f}, y={bbox.get('y', 'N/A'):.1f}, "
                  f"w={bbox.get('width', 'N/A'):.1f}, h={bbox.get('height', 'N/A'):.1f}")

In [0]:
print("="*80)
print("VERIFICATION & ASSERTIONS")
print("="*80)

# Track assertion results
assertions_passed = 0
assertions_total = 0

# Assertion 1: At least one entity detected
assertions_total += 1
if len(entities) > 0:
    print(f"\n✓ PASS: Entities detected ({len(entities)} found)")
    assertions_passed += 1
else:
    print(f"\n❌ FAIL: No entities detected")

# Assertion 2: Employee name detected
assertions_total += 1
employee_names = [e for e in entities if "employee" in e.get("entity_type", "").lower() 
                  or "person" in e.get("entity_type", "").lower()
                  or "name" in e.get("entity_type", "").lower()]
if len(employee_names) > 0:
    print(f"✓ PASS: Employee name detected ({len(employee_names)} found)")
    for name in employee_names:
        print(f"     - '{name.get('text')}'")
    assertions_passed += 1
else:
    print(f"❌ FAIL: No employee name detected")

# Assertion 3: Medicare number detected
assertions_total += 1
medicare_numbers = [e for e in entities if "medicare" in e.get("entity_type", "").lower()]
if len(medicare_numbers) > 0:
    print(f"✓ PASS: Medicare number detected ({len(medicare_numbers)} found)")
    for num in medicare_numbers:
        print(f"     - '{num.get('text')}'")
    assertions_passed += 1
else:
    print(f"❌ FAIL: No Medicare number detected")

# Assertion 4: All entities have bounding boxes
assertions_total += 1
entities_with_bbox = [e for e in entities if e.get("bounding_box") and 
                      all(k in e["bounding_box"] for k in ["x", "y", "width", "height"])]
if len(entities_with_bbox) == len(entities):
    print(f"✓ PASS: All entities have bounding boxes ({len(entities_with_bbox)}/{len(entities)})")
    assertions_passed += 1
else:
    print(f"❌ FAIL: Not all entities have bounding boxes ({len(entities_with_bbox)}/{len(entities)})")

# Assertion 5: Processing completed in reasonable time
assertions_total += 1
if elapsed_time < 60:
    print(f"✓ PASS: Processing time acceptable ({elapsed_time:.2f}s < 60s)")
    assertions_passed += 1
else:
    print(f"⚠ WARNING: Processing time slow ({elapsed_time:.2f}s >= 60s)")

# Final summary
print("\n" + "="*80)
print(f"ASSERTIONS: {assertions_passed}/{assertions_total} PASSED")
print("="*80)

if assertions_passed == assertions_total:
    print("\n🎉 INTEGRATION TEST: SUCCESS 🎉")
else:
    print(f"\n⚠️ INTEGRATION TEST: PARTIAL FAILURE ({assertions_total - assertions_passed} failed) ⚠️")